# 국민은행_토스앱 리뷰 1년치 수집_저장_텍스트분석_mini_project
* 1. google play에서 국민은행 스타뱅킹앱, 토스앱의 사용자 리뷰를 1년치 수집
* 2. 수집한 리뷰를 mysql에 저장
* 3. 수집된 데이터로 긍정/부정 리뷰 비율 분석(막대 그래프 비교)
* 4. 워드클라우드 분석
* 5. LDA 토픽 모델링 분석(최적 k 탐색 포함)
* 6. word2vec 생성 후 T-sne 로 시각화
* 7. 분석 결과를 통해 얻을 수 있는 인사이트(문제점, 개선방안) 보고서 작성

# 데이터 1년치 수집

In [1]:
# !pip install selenium webdriver_manager tqdm sqlalchemy pymysql

In [23]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from datetime import datetime, timedelta
# import datetime
import pandas as pd
import time
from tqdm import tqdm
from dbio import to_db

# 1. 은행 별로 1년치 실행 함수

## 아래 함수 : 브라우저 제어 -> 리뷰창 진입 -> 최신순 정렬 -> 날짜 비교 스크롤
## 데이터 확보 라는 일련의 과정을 자동화한 것

이 코드가 다 돌아가면, 사용자님은 수천 개의 리뷰가 담긴 reviews 리스트를 얻게 되고, 이걸 **데이터 가공 함수(review_extraction)** 에 넘겨주기만 하면 됨

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from datetime import datetime, timedelta
import time
import shutil

# 은행 별로 1년치 실행 함수
def bank_app_reviews(bank, year=1):
    options = Options()

    # ✅ headless에서는 detach 의미 거의 없음(끄는 게 안전)
    # options.add_experimental_option("detach", True)

    # ✅ DevToolsActivePort 예방용 "검증된" 옵션 조합
#     options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--remote-debugging-port=9222")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=ko-KR")

    # (선택) 크래시 감소
    options.add_argument("--disable-features=VizDisplayCompositor")


    # ✅ Chrome/Chromium 바이너리 경로 자동 지정(환경마다 필요)
    chrome_bin = (shutil.which("google-chrome")
                  or shutil.which("google-chrome-stable")
                  or shutil.which("chromium")
                  or shutil.which("chromium-browser"))
    if chrome_bin:
        options.binary_location = chrome_bin

    # 크롬 브라우저 실행 및 주소 접속
    # 사람의 행동: 크롬 아이콘을 더블 클릭해서 켠 다음,
    # 주소창에 해당 은행(토스나 국민은행)의 앱 상세 페이지 주소를 치고 엔터를 누르는 것과 같음
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    url = f"https://play.google.com/store/apps/details?id={bank}"
    driver.get(url)

    wait = WebDriverWait(driver, 15)

    # 리뷰 창 열기 및 정렬 바꾸기 (가장 중요!)
    # 평점 및 리뷰 옆의 -> '평점 및 리뷰 자세히 알아보기' 화살표 버튼 클릭
    button = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "button[aria-label*='평점 및 리뷰 자세히 알아보기']"))
    )
    button.click()

    # 최신순 정렬 버튼 클릭해서 '최신순'으로 변경
    # sortBy_1은 페이지/언어/레이아웃 따라 바뀔 수 있어 예외처리 권장
    button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#sortBy_1")))
    button.click()
    time.sleep(2)

    # 최신 클릭
    button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "span[aria-label*='최신']")))
    button.click()
    time.sleep(2)

    # 목표 날짜 정하기
    # 사람의 행동: "오늘이 2024년 3월이니까,
    # 1년 전인 2023년 3월까지만 긁어오면 되겠네"라고 종료 지점을 머릿속에 저장하는 단계
    today = datetime.today()
    end_date = today - timedelta(days=year * 365)
    print(today, end_date)
    current_date = None
    pre_n_reviews = 0
    stuck = 0
    
    # 무한 스크롤링 -  리뷰 로딩하기
    # 사람의 행동: 플레이스토어는 마우스 휠을 내릴 때마다 새로운 리뷰가 나타남
    # 로봇이 휠을 굴리고 -> 새로 뜬 리뷰 확인하고 -> 날짜 보고를 무한 반복하는 과정
    while True: # 목표 날짜 나올 때까지 무한 반복
        # 1. 스크롤을 아래로 2000픽셀만큼 내리기
        driver.execute_script("document.querySelector('.fysCi.Vk3ZVd').scrollBy(0, 2000)")
        time.sleep(0.8)
        
        # 2. 지금 화면에 보이는 리뷰들 다 체크하기
        reviews = driver.find_elements(By.CSS_SELECTOR, ".RHo1pe")
        n_reviews = len(reviews)

        # 3. 맨 마지막 리뷰의 날짜 확인하기
        try:
            date_text = reviews[-1].find_element(By.CSS_SELECTOR, "div.Jx4nYe > span.bp9Aid").text
            date_text = date_text.replace("년 ", "-").replace("월 ", "-").replace("일", "")
            current_date = datetime.strptime(date_text, "%Y-%m-%d")
        except Exception:
            # 날짜를 못 읽는 경우가 있을 수 있어 계속 진행
            pass

        print(f"{bank} 현재 리뷰 날짜: {current_date}, 리뷰수: {n_reviews} ", end="\r")

        # 목표 기간 도달하면 종료
        if current_date and current_date < end_date:
            print(f"\n{bank}: {year}년 이전 리뷰 도달. 종료.")
            break

    return driver, reviews

# 2. 데이터 가공 및 DB 저장 함수

In [1]:
# 데이터 가공 및 DB 저장 함수

def review_extraction(driver, reviews):    
    all_result = [] # 추출된 개별 리뷰 데이터를 순차적으로 저장할 빈 리스트 객체
    
    # 이전 함수에서 수집한 reviews 리스트(리뷰 요소들의 집합)를 하나씩 순회
    for review in tqdm(reviews):
        # 리뷰 작성일 추출 및 데이터 타입 변환
        # 리뷰일
        review_date = review.find_element(By.CSS_SELECTOR, "div.Jx4nYe > span.bp9Aid").text
        # 문자열 치환(replace) - 날짜형 데이터 타입으로 변환
        review_date = review_date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
        # 객체화 (strptime) - 문자열(String)을 날짜 객체(Datetime object)로 변환
        review_date = datetime.strptime(review_date, "%Y-%m-%d")
        # 별점
        rating = review.find_element(By.CSS_SELECTOR, "div.Jx4nYe > div").get_attribute("aria-label").split()[3][0]
        
        # 리뷰 본문 및 답변 추출 (예외 처리)
        # 사용자 리뷰 텍스트 내용 추출
        user_review = review.find_element(By.CSS_SELECTOR, "div.h3YV2d").text

        try:
            # 업체 댓글
            reply = review.find_element(By.CSS_SELECTOR, "div.ocpBU div.ras4vb > div").get_attribute("innerHTML")
        except:
            reply = None
        
        # 구조화된 데이터 생성 및 DB 전송
        result = (review_date, rating, user_review, reply)
        all_result.append(result)
    
    # Pandas 데이터프레임으로 변환
    result_df = pd.DataFrame(all_result, columns=['리뷰일', '평점', '사용자리뷰', '업체답변'])
    
# 엑셀(csv) 파일로 보고 싶으면: result_df.to_csv(f"./scraping_results/{bank}_review_result.csv", index=False, encoding="utf-8-sig")
    to_db("bank_app_reviews04", bank, result_df)
    print(f"{bank} 리뷰 저장 완료")
    driver.close()  
    

# 3. 데이터 수집, 저장 함수를 최종 실행하는 셀
아래 코드는 앞서 정의한 bank_app_reviews(수집 로봇)와 review_extraction(보석 캐기 및 저장) 함수들을 실제로 가동시키는 '최종 실행 스위치' 셀임

In [29]:
bank_app_list = ['viva.republica.toss', 'com.kbstar.kbbank']

for bank in bank_app_list:
    driver, reviews = bank_app_reviews(bank, 1)
    review_extraction(driver, reviews)

2026-03-05 16:28:57.549640 2025-03-05 16:28:57.549640
viva.republica.toss 현재 리뷰 날짜: 2025-03-04 00:00:00, 리뷰수: 5820 
viva.republica.toss: 1년 이전 리뷰 도달. 종료.


100%|██████████████████████████████████████████████████████████████████████████████| 5820/5820 [03:44<00:00, 25.98it/s]


bank_app_reviews04 데이터베이스 확인/생성 완료
bank_app_reviews04.viva.republica.toss 데이터 저장 완료(append)
viva.republica.toss 리뷰 저장 완료
2026-03-05 17:22:06.666220 2025-03-05 17:22:06.666220
com.kbstar.kbbank 현재 리뷰 날짜: 2025-03-05 00:00:00, 리뷰수: 4180 
com.kbstar.kbbank: 1년 이전 리뷰 도달. 종료.


100%|██████████████████████████████████████████████████████████████████████████████| 4180/4180 [02:38<00:00, 26.44it/s]


bank_app_reviews04 데이터베이스 확인/생성 완료
bank_app_reviews04.com.kbstar.kbbank 데이터 저장 완료(append)
com.kbstar.kbbank 리뷰 저장 완료
